In [1]:
import os
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import BartForConditionalGeneration, AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.modeling_outputs import BaseModelOutput
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import evaluate
import wandb
from torch.amp import autocast, GradScaler
import nltk
import sys

nltk.download('punkt', quiet=True)

class Config:
    def __init__(self):
        # Resolve paths
        self.base = '.'
        self.excel_path = os.path.join(self.base, '../../data/VideoQuestions.xlsx')
        self.sheet_name = 'Dropped Instances With NER Scor'
        self.clip_path = os.path.join(self.base, '../../embeddings/QCLIP')
        self.eff_path = os.path.join(self.base, '../../embeddings/QEfficient')
        
        # Architecture Settings
        self.bart_model = 'facebook/bart-base'
        self.t5_model = "mrm8488/t5-base-finetuned-question-generation-ap"
        self.d_model = 768
        self.num_heads = 8
        
        # Training Parameters
        self.max_input_length = 1024
        self.max_target_length = 64
        self.batch_size = 4
        self.grad_accum_steps = 4
        self.num_epochs = 100
        self.lr = 3e-5
        self.weight_decay = 0.01
        self.dropout = 0.1
        self.patience = 10
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.seed = 42

cfg = Config()
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
print(f"Device: {cfg.device}")

C:\Users\tejes\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [2]:
class AdvancedVideoQADataset(Dataset):
    def __init__(self, df, tokenizer, cfg):
        self.samples = []
        print(f"Loading multimodal features from: {cfg.clip_path}")
        for _, row in tqdm(df.iterrows(), total=len(df)):
            v_id, st = row['video_id'], row['start_time']
            c_p = os.path.join(cfg.clip_path, str(v_id), f"{float(st):.1f}", 'embeddings.npy')
            e_p = os.path.join(cfg.eff_path, str(v_id), f"{float(st):.1f}", 'embeddings.npy')
            
            if os.path.exists(c_p) and os.path.exists(e_p):
                c, e = np.load(c_p)[:21], np.load(e_p)[:21]
                if len(c) < 21:
                    c = np.pad(c, ((0, 21-len(c)), (0, 0)))
                    e = np.pad(e, ((0, 21-len(e)), (0, 0)))
                
                input_text = f"{row['chapter title']} {row['video_title']} {row['summary']}"
                
                inputs = tokenizer(input_text, max_length=cfg.max_input_length, 
                                  padding='max_length', truncation=True, return_tensors='pt')
                
                target = tokenizer(str(row['Questions']), max_length=cfg.max_target_length, 
                                  padding='max_length', truncation=True, return_tensors='pt')
                
                labels = target['input_ids'].squeeze(0)
                labels[labels == tokenizer.pad_token_id] = -100
                
                self.samples.append({
                    'input_ids': inputs['input_ids'].squeeze(0),
                    'attention_mask': inputs['attention_mask'].squeeze(0),
                    'clip': torch.from_numpy(c).float(),
                    'eff': torch.from_numpy(e).float(),
                    'labels': labels
                })
        
        if not self.samples:
            print("WARNING: Zero valid samples found. Please check your data paths.")
        else:
            print(f"Dataset ready: {len(self.samples)} valid samples.")

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]

tokenizer = AutoTokenizer.from_pretrained(cfg.bart_model)
df = pd.read_excel(cfg.excel_path, sheet_name=cfg.sheet_name)

df = df[df['Questions'].notnull()]
train_df, val_df = train_test_split(df, test_size=0.1, random_state=cfg.seed)

train_loader = DataLoader(AdvancedVideoQADataset(train_df, tokenizer, cfg), batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(AdvancedVideoQADataset(val_df, tokenizer, cfg), batch_size=cfg.batch_size)

Loading multimodal features from: .\../../embeddings/QCLIP


100%|██████████| 1370/1370 [00:21<00:00, 63.15it/s]


Dataset ready: 1226 valid samples.
Loading multimodal features from: .\../../embeddings/QCLIP


100%|██████████| 153/153 [00:01<00:00, 77.78it/s]

Dataset ready: 130 valid samples.


In [3]:
class BaselineVideoBART(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.bart = BartForConditionalGeneration.from_pretrained(cfg.bart_model)
        self.clip_proj = nn.Linear(768, cfg.d_model)
        self.eff_proj = nn.Linear(1792, cfg.d_model)
        self.cross_attn = nn.MultiheadAttention(cfg.d_model, cfg.num_heads, batch_first=True)
        self.ln_fusion = nn.LayerNorm(cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)
        self.multimodal_cross_attn = nn.MultiheadAttention(cfg.d_model, cfg.num_heads, batch_first=True)
        self.ln_multimodal = nn.LayerNorm(cfg.d_model)
        
    def get_video_context(self, clip_feats, eff_feats):
        c, e = self.clip_proj(clip_feats), self.eff_proj(eff_feats)
        fused, _ = self.cross_attn(query=e, key=c, value=c)
        fused = self.ln_fusion(fused + e) 
        return fused

    def forward(self, clip_feats, eff_feats, input_ids, attention_mask, labels=None):
        text_outputs = self.bart.model.encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_hidden = text_outputs.last_hidden_state
        video_ctx = self.get_video_context(clip_feats, eff_feats)
        
        mm_fused, _ = self.multimodal_cross_attn(query=text_hidden, key=video_ctx, value=video_ctx)
        final_hidden = self.ln_multimodal(text_hidden + self.dropout(mm_fused))
        
        return self.bart(encoder_outputs=BaseModelOutput(last_hidden_state=final_hidden), labels=labels, return_dict=True)

    def generate(self, clip_feats, eff_feats, input_ids, attention_mask, **kwargs):
        text_outputs = self.bart.model.encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_hidden = text_outputs.last_hidden_state
        video_ctx = self.get_video_context(clip_feats, eff_feats)
        
        mm_fused, _ = self.multimodal_cross_attn(query=text_hidden, key=video_ctx, value=video_ctx)
        final_hidden = self.ln_multimodal(text_hidden + self.dropout(mm_fused))
        
        return self.bart.generate(encoder_outputs=BaseModelOutput(last_hidden_state=final_hidden), **kwargs)

model = BaselineVideoBART(cfg).to(cfg.device)

Loading weights: 100%|██████████| 259/259 [00:00<00:00, 4733.11it/s]


In [4]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=2.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, output1, output2, label=None):
        # Extract encoder hidden states
        if hasattr(output1, 'encoder_last_hidden_state'):
            tensor1 = output1.encoder_last_hidden_state
        else:
            tensor1 = output1[0]
            
        if hasattr(output2, 'encoder_last_hidden_state'):
            tensor2 = output2.encoder_last_hidden_state
        else:
            tensor2 = output2[0]

        # Match sequence length
        if tensor1.size(1) != tensor2.size(1):
            max_seq = max(tensor1.size(1), tensor2.size(1))
            if tensor1.size(1) < max_seq:
                tensor1 = F.pad(tensor1, (0, 0, 0, max_seq - tensor1.size(1)))
            if tensor2.size(1) < max_seq:
                tensor2 = F.pad(tensor2, (0, 0, 0, max_seq - tensor2.size(1)))

        # Match hidden dimension
        if tensor1.size(2) != tensor2.size(2):
            max_hid = max(tensor1.size(2), tensor2.size(2))
            if tensor1.size(2) < max_hid:
                tensor1 = F.pad(tensor1, (0, max_hid - tensor1.size(2)))
            if tensor2.size(2) < max_hid:
                tensor2 = F.pad(tensor2, (0, max_hid - tensor2.size(2)))
        
        return F.mse_loss(tensor1, tensor2)

model_2 = AutoModelForSeq2SeqLM.from_pretrained(cfg.t5_model).to(cfg.device)
tokenizer_2 = AutoTokenizer.from_pretrained(cfg.t5_model)
contrastive_loss_fn = ContrastiveLoss()

Loading weights: 100%|██████████| 260/260 [00:00<00:00, 3619.93it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [5]:
rouge = evaluate.load('rouge', quiet=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scaler = GradScaler('cuda')
best_rouge_l = 0

wandb.init(project="VQG-Baseline", name=f"Baseline-BART-Contrastive")

for epoch in range(cfg.num_epochs):
    model.train()
    total_train_loss, train_steps = 0, 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for i, batch in enumerate(pbar):
        c, e, l = batch['clip'].to(cfg.device), batch['eff'].to(cfg.device), batch['labels'].to(cfg.device)
        in_ids, attn_mask = batch['input_ids'].to(cfg.device), batch['attention_mask'].to(cfg.device)
        
        with autocast('cuda'):
            outputs = model(c, e, in_ids, attn_mask, l)
            loss_ce = outputs.loss
            
            inputs_2_str = tokenizer.batch_decode(in_ids, skip_special_tokens=True)
            encoded_inputs_2 = tokenizer_2(inputs_2_str, max_length=cfg.max_input_length, 
                                          truncation=True, padding=True, return_tensors="pt").to(cfg.device)
            
            with torch.no_grad():
                model_2_outputs = model_2(input_ids=encoded_inputs_2['input_ids'], 
                                         attention_mask=encoded_inputs_2['attention_mask'],
                                         decoder_input_ids=encoded_inputs_2['input_ids'])
            
            loss_contrastive = contrastive_loss_fn(outputs, model_2_outputs)
            loss = (loss_ce + loss_contrastive) / max(1, cfg.grad_accum_steps)
            
        scaler.scale(loss).backward()
        if (i + 1) % cfg.grad_accum_steps == 0:
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
            
        total_train_loss += loss.item() * cfg.grad_accum_steps
        train_steps += 1
        pbar.set_postfix({'loss': total_train_loss / max(1, train_steps)})
    
    # Validation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            c, e, l = batch['clip'].to(cfg.device), batch['eff'].to(cfg.device), batch['labels'].to(cfg.device)
            in_ids, attn_mask = batch['input_ids'].to(cfg.device), batch['attention_mask'].to(cfg.device)
            
            gen_ids = model.generate(c, e, in_ids, attn_mask, max_length=cfg.max_target_length, num_beams=5)
            all_preds.extend(tokenizer.batch_decode(gen_ids, skip_special_tokens=True))
            all_labels.extend([tokenizer.decode(g[g != -100], skip_special_tokens=True) for g in l])

    r_res = rouge.compute(predictions=all_preds, references=all_labels)
    print(f"Epoch {epoch+1} | Loss: {total_train_loss/train_steps:.4f} | RougeL: {r_res['rougeL']:.4f}")
    wandb.log({"epoch": epoch+1, "train_loss": total_train_loss/train_steps, "rougeL": r_res['rougeL']})
    
    if r_res['rougeL'] > best_rouge_l:
        best_rouge_l = r_res['rougeL']
        torch.save(model.state_dict(), 'BaselineBart.pt')

wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\tejes\_netrc.
wandb: Currently logged in as: tejeshwarsingh1205 (tejeshwarsingh1205-indian-institute-of-technology-patna) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1: 100%|██████████| 307/307 [01:07<00:00,  4.55it/s, loss=8.36]


Epoch 1 | Loss: 8.3564 | RougeL: 0.0051


Epoch 2: 100%|██████████| 307/307 [01:50<00:00,  2.78it/s, loss=6.44]


Epoch 2 | Loss: 6.4373 | RougeL: 0.1420


...

Epoch 35 | Loss: 2.1599 | RougeL: 0.4159


Epoch 36: 100%|██████████| 307/307 [01:15<00:00,  4.09it/s, loss=2.15]


Epoch 36 | Loss: 2.1533 | RougeL: 0.4076


Epoch 37:  82%|████████▏ | 251/307 [00:59<00:13,  4.22it/s, loss=2.12]


KeyboardInterrupt: 

In [6]:
# Final testing and metrics
print("Loading best model for final testing...")
model.load_state_dict(torch.load('BaselineBart.pt'))
model.eval()

# Load metrics
rouge = evaluate.load('rouge', quiet=True)
bleu = evaluate.load('bleu', quiet=True)
meteor = evaluate.load('meteor', quiet=True)
bertscore = evaluate.load('bertscore', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(val_loader, desc="Testing"):
        c, e, l = batch['clip'].to(cfg.device), batch['eff'].to(cfg.device), batch['labels'].to(cfg.device)
        in_ids, attn_mask = batch['input_ids'].to(cfg.device), batch['attention_mask'].to(cfg.device)
        
        gen_ids = model.generate(c, e, in_ids, attn_mask, max_length=cfg.max_target_length, num_beams=5)
        all_preds.extend(tokenizer.batch_decode(gen_ids, skip_special_tokens=True))
        all_labels.extend([tokenizer.decode(g[g != -100], skip_special_tokens=True) for g in l])

# Compute metrics
r_res = rouge.compute(predictions=all_preds, references=all_labels)
b_res = bleu.compute(predictions=all_preds, references=[[l] for l in all_labels])
b1_res = bleu.compute(predictions=all_preds, references=[[l] for l in all_labels], max_order=1)
m_res = meteor.compute(predictions=all_preds, references=all_labels)
bs_res = bertscore.compute(predictions=all_preds, references=all_labels, lang="en")
bs_f1 = np.mean(bs_res['f1'])

def calculate_distinct_n(predictions, n):
    total_ngrams = 0
    unique_ngrams = set()
    for pred in predictions:
        tokens = nltk.word_tokenize(pred.lower())
        ngrams = list(nltk.ngrams(tokens, n))
        total_ngrams += len(ngrams)
        unique_ngrams.update(ngrams)
    return len(unique_ngrams) / total_ngrams if total_ngrams > 0 else 0

dist1 = calculate_distinct_n(all_preds, 1)
dist2 = calculate_distinct_n(all_preds, 2)

final_metrics = {
    "rougeL": r_res['rougeL'],
    "bleu1": b1_res['bleu'],
    "bleu": b_res['bleu'],
    "meteor": m_res['meteor'],
    "bert_score": bs_f1,
    "distinct1": dist1,
    "distinct2": dist2
}

print("\nFinal Test Metrics:")
print(json.dumps(final_metrics, indent=4))


Loading best model for final testing...


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\tejes\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\tejes\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\tejes\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 4582.02it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias             


Final Test Metrics:
{
    "rougeL": 0.42240606761979405,
    "bleu1": 0.4168452061925679,
    "bleu": 0.11563674826920466,
    "meteor": 0.3806035160109971,
    "bert_score": 0.8898049079454862,
    "distinct1": 0.33218291630716135,
    "distinct2": 0.7453838678328474
}
